# ClearBank — Análise Financeira com Python

Notebook do desafio final: leitura e validação de `transacoes.csv`, métricas mensais, transações suspeitas, relatório no terminal e exportação para `relatorio.json`.

## Instruções

1. Coloque o arquivo `transacoes.csv` na mesma pasta deste notebook.
2. Execute as células **em ordem** (Run All).
3. A célula de execução principal gera o relatório no terminal e o arquivo `relatorio.json`.

## Colunas esperadas no CSV

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| id | inteiro | Identificador único |
| data | texto | Formato `AAAA-MM-DD` |
| cliente_id | texto | Código do cliente (obrigatório) |
| tipo | texto | `credito` ou `debito` |
| valor | decimal | Maior que 0 |
| descricao | texto | Descrição livre |
| categoria | texto | Ex.: salario, compra, transferencia |

In [1]:
import csv
import json
from datetime import datetime, date
from collections import defaultdict

LIMITE_SUSPEITO = 10000.00
TIPOS_VALIDOS = {"credito", "debito"}
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"

print("Constantes carregadas.")
print(f"LIMITE_SUSPEITO = {LIMITE_SUSPEITO:.2f}")

Constantes carregadas.
LIMITE_SUSPEITO = 10000.00


## Leitura do CSV

Usa o módulo nativo `csv.DictReader`. Se o arquivo não existir, captura `FileNotFoundError` e retorna lista vazia.

In [2]:
def ler_transacoes(caminho: str) -> list[dict]:
    """Read CSV rows as dictionaries. Returns [] if the file is missing."""
    try:
        with open(caminho, encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho}")
        return []


linhas_brutas = ler_transacoes(ARQUIVO_CSV)
print(f"Linhas lidas: {len(linhas_brutas)}")
if linhas_brutas:
    print("Primeira linha:", linhas_brutas[0])

Linhas lidas: 30
Primeira linha: {'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salário janeiro', 'categoria': 'salario'}


## Validação de campos

Funções pequenas para `id`, `valor` e `data`. Erros de conversão são capturados e a linha é descartada depois.

In [3]:
def validar_id(texto: str | None) -> int | None:
    if texto is None:
        return None
    texto = str(texto).strip()
    if not texto:
        return None
    try:
        return int(texto)
    except ValueError:
        return None


def validar_valor(texto: str | None) -> float | None:
    if texto is None:
        return None
    try:
        valor = float(str(texto).strip())
    except (ValueError, AttributeError):
        return None
    if valor <= 0:
        return None
    return valor


def converter_data(texto: str | None) -> datetime | None:
    if texto is None:
        return None
    try:
        return datetime.strptime(str(texto).strip(), "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None


print("id válido:", validar_id("10"))
print("id inválido:", validar_id("abc"))
print("valor válido:", validar_valor("180.50"))
print("valor inválido:", validar_valor("abc"), validar_valor("-1"))
print("data válida:", converter_data("2026-01-05"))
print("data inválida:", converter_data("2026/01/05"))

id válido: 10
id inválido: None
valor válido: 180.5
valor inválido: None None
data válida: 2026-01-05 00:00:00
data inválida: None


## Validação da transação e limpeza

Descarta linhas inválidas sem interromper o processamento. IDs duplicados também são descartados.

In [4]:
def validar_transacao(linha: dict) -> dict | None:
    """Return a cleaned transaction or None if the row is invalid."""
    try:
        transacao_id = validar_id(linha.get("id"))
        cliente_id = str(linha.get("cliente_id") or "").strip()
        data_obj = converter_data(linha.get("data"))
        tipo = str(linha.get("tipo") or "").strip().lower()
        valor = validar_valor(linha.get("valor"))
    except (KeyError, AttributeError, TypeError):
        return None

    if transacao_id is None or not cliente_id or data_obj is None:
        return None
    if tipo not in TIPOS_VALIDOS or valor is None:
        return None

    return {
        "id": transacao_id,
        "data": data_obj,
        "mes": data_obj.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": str(linha.get("descricao") or "").strip(),
        "categoria": str(linha.get("categoria") or "").strip(),
    }


def limpar_transacoes(linhas: list[dict]) -> tuple[list[dict], int]:
    validas: list[dict] = []
    ids_vistos: set[int] = set()
    invalidas = 0

    for linha in linhas:
        registro = validar_transacao(linha)
        if registro is None or registro["id"] in ids_vistos:
            invalidas += 1
            continue
        ids_vistos.add(registro["id"])
        validas.append(registro)

    return validas, invalidas


def exibir_resumo_limpeza(total_lidas: int, validas: int, invalidas: int) -> None:
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {validas}")
    print(f"Linhas inválidas: {invalidas}")


validas_teste, invalidas_teste = limpar_transacoes(linhas_brutas)
exibir_resumo_limpeza(len(linhas_brutas), len(validas_teste), invalidas_teste)

Total de linhas lidas: 30
Linhas válidas: 20
Linhas inválidas: 10


## Período analisado (datetime)

In [5]:
def calcular_periodo(transacoes: list[dict]) -> dict:
    if not transacoes:
        return {"inicio": None, "fim": None, "dias": 0}

    datas = [item["data"] for item in transacoes]
    inicio = min(datas)
    fim = max(datas)
    return {
        "inicio": inicio.strftime("%Y-%m-%d"),
        "fim": fim.strftime("%Y-%m-%d"),
        "dias": (fim - inicio).days,
    }


periodo_teste = calcular_periodo(validas_teste)
print(periodo_teste)

{'inicio': '2026-01-05', 'fim': '2026-04-22', 'dias': 107}


## Métricas mensais e transações suspeitas

In [6]:
def agrupar_por_mes(transacoes: list[dict]) -> dict[str, list[dict]]:
    grupos: dict[str, list[dict]] = defaultdict(list)
    for item in transacoes:
        grupos[item["mes"]].append(item)
    return dict(sorted(grupos.items()))


def calcular_metricas_mes(lista: list[dict]) -> dict:
    valores = [item["valor"] for item in lista]
    total_credito = sum(item["valor"] for item in lista if item["tipo"] == "credito")
    total_debito = sum(item["valor"] for item in lista if item["tipo"] == "debito")
    quantidade = len(lista)
    media = (sum(valores) / quantidade) if quantidade else 0.0
    return {
        "quantidade": quantidade,
        "total_credito": round(total_credito, 2),
        "total_debito": round(total_debito, 2),
        "saldo": round(total_credito - total_debito, 2),
        "media": round(media, 2),
        "maior_valor": round(max(valores), 2) if valores else 0.0,
        "menor_valor": round(min(valores), 2) if valores else 0.0,
    }


def identificar_suspeitas(transacoes: list[dict]) -> list[dict]:
    suspeitas = []
    for item in transacoes:
        if item["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": item["id"],
                "cliente_id": item["cliente_id"],
                "data": item["data"].strftime("%Y-%m-%d"),
                "valor": round(item["valor"], 2),
            })
    return suspeitas


grupos_teste = agrupar_por_mes(validas_teste)
print("Meses:", list(grupos_teste.keys()))
print("Suspeitas:", identificar_suspeitas(validas_teste))

Meses: ['2026-01', '2026-02', '2026-03', '2026-04']
Suspeitas: [{'id': 8, 'cliente_id': 'CLI003', 'data': '2026-02-14', 'valor': 15000.0}, {'id': 14, 'cliente_id': 'CLI004', 'data': '2026-03-20', 'valor': 12000.0}, {'id': 20, 'cliente_id': 'CLI005', 'data': '2026-04-22', 'valor': 18500.0}]


## Relatório e exportação JSON

In [7]:
def gerar_relatorio(validas: list[dict], invalidas: int) -> dict:
    grupos = agrupar_por_mes(validas)
    resumo_mensal = {
        mes: calcular_metricas_mes(lista) for mes, lista in grupos.items()
    }
    return {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": len(validas),
        "total_transacoes_invalidas": invalidas,
        "periodo": calcular_periodo(validas),
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": identificar_suspeitas(validas),
    }


def salvar_json(relatorio: dict, caminho: str) -> bool:
    try:
        with open(caminho, "w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
        return True
    except OSError as erro:
        print(f"Não foi possível salvar {caminho}: {erro}")
        return False


relatorio_teste = gerar_relatorio(validas_teste, invalidas_teste)
print("Chaves do relatório:", list(relatorio_teste.keys()))
print("Total válidas:", relatorio_teste["total_transacoes_validas"])

Chaves do relatório: ['gerado_em', 'total_transacoes_validas', 'total_transacoes_invalidas', 'periodo', 'resumo_mensal', 'transacoes_suspeitas']
Total válidas: 20


## Exibição formatada no terminal

In [8]:
def formatar_moeda(valor: float) -> str:
    texto = f"{valor:,.2f}"
    return "R$ " + texto.replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio: dict) -> None:
    periodo = relatorio["periodo"]
    print("===== RELATÓRIO MENSAL =====")
    print()
    print(f"Gerado em: {relatorio['gerado_em']}")
    print(
        f"Período analisado: {periodo['inicio']} → {periodo['fim']} "
        f"({periodo['dias']} dias)"
    )
    print(f"Transações válidas: {relatorio['total_transacoes_validas']}")
    print(f"Transações inválidas: {relatorio['total_transacoes_invalidas']}")
    print()

    for mes, metricas in relatorio["resumo_mensal"].items():
        print(f"Mês: {mes}")
        print(f"  Transações: {metricas['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(metricas['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(metricas['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(metricas['saldo'])}")
        print(f"  Média:         {formatar_moeda(metricas['media'])}")
        print(f"  Maior valor:   {formatar_moeda(metricas['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(metricas['menor_valor'])}")
        print()

    print("===== TRANSAÇÕES SUSPEITAS =====")
    suspeitas = relatorio["transacoes_suspeitas"]
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
        return
    for item in suspeitas:
        print(
            f"ID: {item['id']} | Cliente: {item['cliente_id']} | "
            f"Data: {item['data']} | Valor: {formatar_moeda(item['valor'])}"
        )


print(formatar_moeda(3500))
print(formatar_moeda(180.5))

R$ 3.500,00
R$ 180,50


## Célula de execução principal

In [9]:
def main() -> None:
    linhas = ler_transacoes(ARQUIVO_CSV)
    validas, invalidas = limpar_transacoes(linhas)
    print("===== RESUMO DA LIMPEZA =====")
    exibir_resumo_limpeza(len(linhas), len(validas), invalidas)
    print()
    relatorio = gerar_relatorio(validas, invalidas)
    exibir_relatorio(relatorio)
    if salvar_json(relatorio, ARQUIVO_JSON):
        print()
        print(f"Relatório salvo em {ARQUIVO_JSON}")


main()

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 30
Linhas válidas: 20
Linhas inválidas: 10

===== RELATÓRIO MENSAL =====

Gerado em: 2026-10-04
Período analisado: 2026-01-05 → 2026-04-22 (107 dias)
Transações válidas: 20
Transações inválidas: 10

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 455,50
  Saldo:         R$ 4.244,50
  Média:         R$ 1.031,10
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 55,00

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 17.800,00
  Total débito:  R$ 559,90
  Saldo:         R$ 17.240,10
  Média:         R$ 3.671,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 16.300,00
  Total débito:  R$ 525,50
  Saldo:         R$ 15.774,50
  Média:         R$ 3.365,10
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 75,50

Mês: 2026-04
  Transações: 5
  Total crédito: R$ 24.500,00
  Total débito:  R$ 1.100,30
  Saldo:         R$ 23.399,70
  Média:         R$ 5.120,0

## Verificação do arquivo JSON gerado

In [10]:
with open(ARQUIVO_JSON, encoding="utf-8") as arquivo:
    conteudo = json.load(arquivo)

print(json.dumps(conteudo, ensure_ascii=False, indent=2))

{
  "gerado_em": "2026-10-04",
  "total_transacoes_validas": 20,
  "total_transacoes_invalidas": 10,
  "periodo": {
    "inicio": "2026-01-05",
    "fim": "2026-04-22",
    "dias": 107
  },
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 5,
      "total_credito": 4700.0,
      "total_debito": 455.5,
      "saldo": 4244.5,
      "media": 1031.1,
      "maior_valor": 3500.0,
      "menor_valor": 55.0
    },
    "2026-02": {
      "quantidade": 5,
      "total_credito": 17800.0,
      "total_debito": 559.9,
      "saldo": 17240.1,
      "media": 3671.98,
      "maior_valor": 15000.0,
      "menor_valor": 99.9
    },
    "2026-03": {
      "quantidade": 5,
      "total_credito": 16300.0,
      "total_debito": 525.5,
      "saldo": 15774.5,
      "media": 3365.1,
      "maior_valor": 12000.0,
      "menor_valor": 75.5
    },
    "2026-04": {
      "quantidade": 5,
      "total_credito": 24500.0,
      "total_debito": 1100.3,
      "saldo": 23399.7,
      "media": 5120.06,
      "m